# IR Practical 4: SMS Spam Classification using KNN and TF-IDF
**Aim:** Preprocess SMS messages, convert them to TF-IDF features, train a K-Nearest Neighbors classifier, evaluate it, and classify a new message.

The PDF uses an SMS spam dataset with `Msg` and `Label` columns.

In [ ]:
%pip -q install nltk scikit-learn pandas

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    confusion_matrix, classification_report
)

nltk.download("stopwords", quiet=True)

In [ ]:
# Dataset source represented in the PDF.
url = "https://raw.githubusercontent.com/junioralive/india-spam-sms-classification/main/dataset/spam.csv"

data = pd.read_csv(url)
print("Dataset loaded successfully!")
display(data.head())

In [ ]:
print("Shape:", data.shape)

print("\\nColumns:")
print(data.columns.tolist())

print("\\nClass Distribution:")
print(data.iloc[:, 1].value_counts())

In [ ]:
# Rename the first two columns to the names used in the practical.
data = data.rename(columns={
    data.columns[0]: "message",
    data.columns[1]: "label"
})

print(data.head())

In [ ]:
data["label"] = data["label"].astype(str).str.lower().str.strip()
data["label_num"] = data["label"].map({
    "ham": 0,
    "spam": 1
})

# Keep only rows with a recognized class label and a message.
data = data.dropna(subset=["message", "label_num"]).copy()
data["label_num"] = data["label_num"].astype(int)

print(data[["message", "label", "label_num"]].head())
print("\\nClass counts:")
print(data["label"].value_counts())

In [ ]:
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def preprocess_text(text):
    text = str(text).lower()

    # Remove punctuation and digits while retaining letters and whitespace.
    text = re.sub(r"[^a-z\\s]", " ", text)

    # Tokenization
    words = text.split()

    # Stop-word removal
    words = [
        word for word in words
        if word not in stop_words
    ]

    # Stemming
    words = [
        stemmer.stem(word)
        for word in words
    ]

    return " ".join(words)

In [ ]:
data["clean_message"] = data["message"].apply(preprocess_text)

print("Original:")
print(data["message"].iloc[0])

print("\\nAfter Preprocessing:")
print(data["clean_message"].iloc[0])

In [ ]:
X = data["clean_message"]
y = data["label_num"]

print("Input samples:", len(X))
print("Output samples:", len(y))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training feature shape:", X_train_tfidf.shape)
print("Testing feature shape:", X_test_tfidf.shape)

In [ ]:
knn = KNeighborsClassifier(
    n_neighbors=5,
    metric="euclidean"
)

print("K-NN model created successfully!")

In [ ]:
knn.fit(X_train_tfidf, y_train)
print("K-NN model trained successfully!")

In [ ]:
y_pred = knn.predict(X_test_tfidf)

print("Predictions:")
print(y_pred[:20])

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy (%):", accuracy * 100)

In [ ]:
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

print("Precision:", precision)
print("Recall/Sensitivity:", recall)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

In [ ]:
print("Classification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Ham", "Spam"],
    zero_division=0
))

In [ ]:
message = input("Enter an email/SMS message: ")

clean_message = preprocess_text(message)
message_vector = vectorizer.transform([clean_message])
prediction = knn.predict(message_vector)[0]

if prediction == 1:
    print("Prediction: SPAM")
else:
    print("Prediction: HAM / NOT SPAM")